# Dataset and tokenizer exploration
Inspect the exact ToyMovieReview split, tokenizer retention, answer tokens, and SST class balance before fitting directions.

In [ ]:
# In a fresh Colab runtime, uncomment after cloning or uploading the project.
# %pip install -e '.[notebooks]'
from pathlib import Path
from sentiment_manifold.config import ReproductionConfig
from sentiment_manifold.data import load_sst, load_toy_movie_review

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cfg = ReproductionConfig.load(PROJECT_ROOT / 'configs/reproduction.yaml')
toy = load_toy_movie_review(cfg.data.toy_config)
len(toy.train), len(toy.test), toy.train[0]

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

frame = pd.DataFrame([{'split': e.metadata['split'], 'label': e.label, 'word': e.metadata['adjective']} for e in toy.train + toy.test])
sns.countplot(data=frame, x='split', hue='label')
plt.title('ToyMovieReview split before tokenizer filtering');

In [ ]:
# Loading the model downloads weights. Change to 'qwen-0.6b' to audit Qwen.
from sentiment_manifold.devices import resolve_device
from sentiment_manifold.models import CausalLMAdapter

spec = resolve_device('auto')
adapter = CausalLMAdapter.from_pretrained(cfg.model.hub_name, spec)
retained = [e for e in toy.train if adapter.focus_is_single_token(e)]
print(f'Device: {spec.device}; retained train examples: {len(retained)}/{len(toy.train)}')
print({label: adapter.tokenizer(answer, add_special_tokens=False)['input_ids'] for label, answer in toy.answers.items()})

In [ ]:
sst = load_sst(cfg.data.sst_root, cfg.data.sst_split)
pd.Series([example.label for example in sst]).value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'})